In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import os
from pathlib import Path
from tqdm.auto import tqdm # 使用 .auto 版本以在 notebook 中获得更好的进度条
from IPython.display import display # 用于在 notebook 中漂亮地显示 DataFrame
import scanpy as sc

In [ ]:
def process_h5ad_folder(folder_path, region_col, celltype_col):
    """
    读取指定文件夹下的所有h5ad文件，并生成region和celltype的计数DataFrame。
    
    参数:
    folder_path (str): 包含h5ad文件的文件夹路径。
    region_col (str): anndata.obs 中代表区域的列名。
    celltype_col (str): anndata.obs 中代表细胞类型的列名。
    
    返回:
    (pd.DataFrame, pd.DataFrame): 区域计数的DataFrame和细胞类型计数的DataFrame。
    """
    
    # 1. 初始化数据结构
    region_results = {}
    celltype_results = {}
    
    # 2. 查找所有h5ad文件
    h5ad_files = list(Path(folder_path).glob('*.h5ad'))
    
    if not h5ad_files:
        print(f"警告：在 '{folder_path}' 中未找到任何 .h5ad 文件。")
        return pd.DataFrame(), pd.DataFrame()

    print(f"在 '{folder_path}' 中找到 {len(h5ad_files)} 个 .h5ad 文件。开始处理...")

    # 3. 遍历和处理每个文件
    # tqdm.auto 会自动选择适合 notebook 的进度条
    for file_path in tqdm(h5ad_files, desc="Processing h5ad files"):
        filename = file_path.name
        
        try:
            # 读取 h5ad 文件
            adata = ad.read_h5ad(file_path)
            
            # --- 处理 Region ---
            if region_col in adata.obs.columns:
                region_counts = adata.obs[region_col].value_counts().to_dict()
            else:
                print(f"  > 提示: '{filename}' 中未找到 '{region_col}' 列。")
                region_counts = {}
            
            region_results[filename] = region_counts
            
            # --- 处理 Celltype ---
            if celltype_col in adata.obs.columns:
                celltype_counts = adata.obs[celltype_col].value_counts().to_dict()
            else:
                print(f"  > 提示: '{filename}' 中未找到 '{celltype_col}' 列。")
                celltype_counts = {}
            
            celltype_results[filename] = celltype_counts

        except Exception as e:
            print(f"处理 '{filename}' 时出错: {e}. 已跳过。")
            continue
            
    # 4. 生成 DataFrame
    print("\n所有文件处理完毕。正在生成 DataFrame...")
    
    # --- Region DataFrame ---
    df_regions = pd.DataFrame.from_dict(region_results, orient='index')
    df_regions = df_regions.fillna(0).astype(int)
    df_regions.index.name = 'filename' # 设置索引列的名称
    
    # --- Celltype DataFrame ---
    df_celltypes = pd.DataFrame.from_dict(celltype_results, orient='index')
    df_celltypes = df_celltypes.fillna(0).astype(int)
    df_celltypes.index.name = 'filename' # 设置索引列的名称

    return df_regions, df_celltypes

In [ ]:
# --- 1. 配置 ---

# !! 修改为您自己的文件夹路径
DATA_FOLDER = "fetal_brain_hongtaoyu/3dCellBin/NB20241129094951824"  

# !! 修改为您的 anndata.obs 中对应的列名
REGION_COLUMN_NAME = 'regions'
CELLTYPE_COLUMN_NAME = 'celltype'

# 3. 设置输出文件名
OUTPUT_REGION_CSV = 'Output/Data_Summary/h5ad_region_counts_cellbin.csv'
OUTPUT_CELLTYPE_CSV = 'Output/Data_Summary/h5ad_celltype_counts_cellbin.csv'

# --- 2. 执行 ---
df_regions, df_celltypes = process_h5ad_folder(
    folder_path=DATA_FOLDER,
    region_col=REGION_COLUMN_NAME,
    celltype_col=CELLTYPE_COLUMN_NAME
)

In [ ]:
df_regions.sum().sum()

In [ ]:
df_regions.sum()

In [ ]:
df_celltypes.sum()

In [ ]:
# --- 3. 显示和保存结果 ---

if not df_regions.empty:
    print(f"\n--- 区域 (Region) 统计 ---")
    display(df_regions) # 在 notebook 中漂亮地显示
    df_regions.to_csv(OUTPUT_REGION_CSV)
    print(f"区域统计已保存到: {OUTPUT_REGION_CSV}")
else:
    print("未生成区域统计数据。")

if not df_celltypes.empty:
    print(f"\n--- 细胞类型 (Celltype) 统计 ---")
    display(df_celltypes) # 在 notebook 中漂亮地显示
    df_celltypes.to_csv(OUTPUT_CELLTYPE_CSV)
    print(f"细胞类型统计已保存到: {OUTPUT_CELLTYPE_CSV}")
else:
    print("未生成细胞类型统计数据。")

print("\n任务完成。")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. 检查 df_regions 是否存在且非空 ---
if 'df_regions' not in locals() or df_regions.empty:
    print("Error: DataFrame 'df_regions' not found or is empty.")
    print("Please ensure you have run the previous cells to generate the data.")
else:
    print("Generating region count bar plot...")

    # --- 2. 准备数据 (Melt DataFrame) ---
    # 将文件名从索引转换为一个普通的列
    # 然后使用 .melt() 将宽格式数据转换为长格式，更适合 seaborn
    df_regions_melted = df_regions.reset_index().melt(
        id_vars='filename', 
        var_name='Region', 
        value_name='Cell_Count'
    )
    
    # (可选) 过滤掉数量为0的行，使图表更清晰
    # df_regions_melted = df_regions_melted[df_regions_melted['Cell_Count'] > 0]

    # --- 3. 绘图 ---
    # 设置图表大小和风格
    plt.figure(figsize=(12, 7))
    sns.set_style("whitegrid")

    # 使用 seaborn.barplot 绘制柱状图
    # dodge=False 是实现堆叠的关键
    sns.barplot(
        data=df_regions_melted, 
        x='filename', 
        y='Cell_Count', 
        hue='Region', 
        dodge=False, # 关键：设置为 False 实现堆叠柱状图
        palette='viridis' # 选择一个颜色主题
    )

    # --- 4. 美化图表 (使用英文标签) ---
    plt.title('Cell Counts by Region per H5AD File', fontsize=16)
    plt.xlabel('H5AD Filename', fontsize=12)
    plt.ylabel('Cell Count', fontsize=12)
    
    # 旋转 x 轴标签，以防文件名过长重叠
    plt.xticks(rotation=45, ha='right') 
    
    # 调整图例位置
    plt.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 调整布局，防止标签或图例被裁剪
    plt.tight_layout()
    
    # 显示图表
    plt.show()